# MNIST Handwritten Digit Classification with a Neural Network


## 1. Environment Setup


In [ ]:
# Colab: run this cell first. It clones/checks out the JWH branch and loads src.
# The GitHub URL and branch are fixed, so you do not need to type them.
import os
import sys
from pathlib import Path

GIT_URL = "https://github.com/Jungle-12-303/wk13_team2_mnist.git"
BRANCH = "JWH"
REPO_NAME = "wk13_team2_mnist"

if "google.colab" in sys.modules:
    repo_path = Path("/content") / REPO_NAME

    # If the repo already exists, reuse it and pull the latest JWH branch.
    if repo_path.exists():
        os.chdir(repo_path)
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    else:
        os.chdir("/content")
        !git clone -b {BRANCH} {GIT_URL}
        os.chdir(repo_path)

    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)
else:
    src_path = str(Path.cwd() / "src")
    if src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)

# Clear cached modules so Colab does not keep old code from another branch.
for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("Current working directory:", Path.cwd())
print("Current branch:")
!git branch --show-current


## 2. Load Data


In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. Run Tests

Run the test cell below when you want to verify the implementation.
- Main implementation files: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- Example imports: `from activations import ReLU`, `from network import NeuralNetwork`
- Start with one test file, then run all tests when ready.
    - ReLU only: `TEST_TARGET = "tests/test_relu.py"`
    - Filter tests by keyword: `PYTEST_KEYWORD = "backward"`
    - All tests: `TEST_TARGET = "tests/"`


In [ ]:
import subprocess
import sys
from pathlib import Path

# Use the current notebook directory as the repository root.
repo_dir = Path.cwd()

# Start with the test file you are working on.
# Examples: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# Use this to run only tests whose names contain a keyword.
# Example: "backward". Leave empty to run the whole file.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("Working directory:", repo_dir)
print("Command:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\nSelected tests passed.")
else:
    print("\nSome selected tests failed.")


## 4. Create Model and Train


In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)


## 5. Evaluate and Plot Loss


In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

## JWH Branch Setup

Run this cell before running the experiment runner.
It updates the notebook to the latest `JWH` branch and clears import caches.


In [ ]:
import os
import sys
from pathlib import Path

REPO_NAME = "wk13_team2_mnist"
repo_dir = Path.cwd()
if not (repo_dir / "src").exists() and Path(f"/content/{REPO_NAME}/src").exists():
    repo_dir = Path(f"/content/{REPO_NAME}")
os.chdir(repo_dir)

!git fetch origin
!git checkout JWH
!git pull origin JWH

src_path = str(Path.cwd() / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

for module_name in ["training", "network", "losses", "layers", "activations", "optimizers", "data"]:
    sys.modules.pop(module_name, None)

print("Ready on branch JWH")
!git branch --show-current


## Experiment Runner

This cell runs one or more MLP experiment configs and saves `results.csv` and `results.json`.
Start with `configs[:1]` for a quick baseline run. Use all configs when you are ready.


In [ ]:
from data import load_mnist
from training import DEFAULT_EXPERIMENT_CONFIGS, run_experiments, plot_compare_loss_histories

(x_train, y_train), (x_test, y_test) = load_mnist()

# Quick check: run only the baseline experiment.
configs = DEFAULT_EXPERIMENT_CONFIGS[:1]

# Full comparison: uncomment the next line.
# configs = DEFAULT_EXPERIMENT_CONFIGS

results, histories = run_experiments(
    configs,
    x_train, y_train,
    x_test, y_test,
    csv_path="results.csv",
    json_path="results.json",
)

results
